In [ ]:
%pip install --no-deps ../.

In [ ]:
import logging
from invoice_pipeline.common import config
from invoice_pipeline.common.tile import Tile
from pyspark.sql import functions as F
import json

In [ ]:
logging.basicConfig(level=logging.INFO)
LOG = logging.getLogger(__name__)

BUNDLE_NAME = config.get("bundle_name", "invoice_pipeline")
BUNDLE_TARGET = config.get("bundle_target", "dev")
CATALOG_NAME = config.get("catalog_name", "reggie_pierce")
SCHEMA_NAME = config.get("schema_name", f"{BUNDLE_NAME}_{BUNDLE_TARGET}")
TRAINING_TABLE_NAME = config.get("training_table_name", "information_extraction_training")
VOLUME_NAME = config.get("volume_name", "files")

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"


In [ ]:
file_df = spark.read.format("binaryFile").load(VOLUME_PATH).limit(5)
ai_parse_df = file_df.select(
    "path",
    "length",
    F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"),
)

ai_parse_text_df = ai_parse_df.withColumn(
    "text",
    F.expr(
        """
        array_join(
          filter(
            transform(
              variant_get(e, '$.document.elements', 'array<variant>'),
              e -> trim(variant_get(e, '$.content', 'string'))
            ),
            x -> x is not null AND x != ''
          ),
          '\n\n'
        ) AS text
    """
    ),
)
ai_parse_text_df.write.mode("append").saveAsTable(TRAINING_TABLE_NAME)

In [ ]:
def _notebook_context():
    return dbutils.notebook.entry_point.getDbutils().notebook().getContext()


def _api_url():
    return _notebook_context().apiUrl().getOrElse(None)


def _api_token():
    return _notebook_context().apiToken().getOrElse(None)

In [ ]:
import requests


def _headers() -> dict[str, str]:
    return {"Content-Type": "application/json",
            "Authorization": f"Bearer {_api_token()}",
            }


def get_tile(tile_name: str) -> Tile | None:
    url = f"{_api_url()}/api/2.0/tiles"
    params = {
        "filter": f"name_contains={tile_name}"
    }
    response = requests.get(url, headers=_headers(), params=params)
    response.raise_for_status()
    tiles = response.json().get("tiles", [])
    if len(tiles) == 1:
        return Tile(data=tiles[0])
    elif len(tiles) > 1:
        LOG.debug(f"Multiple tiles match - {tile_name}")
    return None


In [ ]:
output_json_schema = get_tile(tile_name="reggie_pierce_invoice_pipeline_information_extraction").output_json_schema
print(json.dumps(output_json_schema, indent=2))